# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading and exploring the FAIR⁲ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is described by a [Croissant schema](https://mlcommons.org/croissant/) at the provided URL.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant JSON-LD file URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Access name and description (use object attributes, not subscripting)
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset URL: {croissant_url}")

## 2. Data Overview
Review available record sets, their `@id`, fields, and their `@id`s. This helps us identify the data structure in the next steps.

In [ ]:
# List all record sets with their `@id` and name
print('Available Record Sets:')
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}, description: {rs.description}")

# Pick a main record set (usually the main data). For this dataset, there is likely one main table.
record_sets = [rs.id for rs in metadata.record_sets]

# Show all fields for each record set
for rs in metadata.record_sets:
    print(f"\nRecord Set: {rs.name} (@id={rs.id})")
    print("Fields:")
    for field in rs.fields:
        print(f"  - @id: {field.id}, name: {field.name}, data_type: {field.data_type}, description: {field.description}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# We'll extract all record sets into separate pandas DataFrames for exploration

dataframes = {}
for rs_id in record_sets:
    print(f"Loading records for Record Set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print(f"  No records found in record set {rs_id}.")
        continue
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}")

In [ ]:
# Pick main record set for exploration. Choose the first available, or customize.
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nMain DataFrame columns for record set `{main_record_set_id}`:")
    print(main_df.columns.tolist())
    display(main_df.head())
else:
    print('No dataframes extracted. Please check the record sets above.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on values in a numeric field.
- Normalizing a numeric field.
- Grouping data by a key attribute.

Please adjust the field `@id`s to those that refer to numeric and categorical fields you want to analyze (see field lists above).

In [ ]:
#---------------
# Customization
#---------------
# Pick relevant field `@id`s for this EDA: (change as needed)

# For demonstration, try to pick a numeric field such as protein MW (molecular weight), peptide_counts, or abundance.
numeric_field_id = None
group_field_id = None

# Guess possible numeric/categorical fields from columns
if dataframes and main_record_set_id:
    cols = dataframes[main_record_set_id].columns.tolist()
    # Try to select MW, coverage, or abundance column by matching names
    for c in cols:
        if 'mw' in c.lower() or 'molecular_weight' in c.lower():
            numeric_field_id = c
        elif 'coverage' in c.lower():
            numeric_field_id = c
        elif 'abundance' in c.lower():
            numeric_field_id = c
        elif 'count' in c.lower():
            numeric_field_id = c
    # For group field, pick e.g. a description or protein name
    for c in cols:
        if 'desc' in c.lower() or 'name' in c.lower() or 'accession' in c.lower():
            group_field_id = c
            break
    # Fallbacks
    if numeric_field_id is None and len(cols) > 1:
        numeric_field_id = cols[1]
    if group_field_id is None and len(cols) > 0:
        group_field_id = cols[0]
    print(f"Chose numeric_field_id: {numeric_field_id}, group_field_id: {group_field_id}")else:
    print('No main dataframe to analyze.')

In [ ]:
if dataframes and main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id].copy()
    # Coerce numeric field
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    
    threshold = df[numeric_field_id].mean()  # Example: use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.1f} (mean value):")
    display(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id (e.g., protein id or name)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print('Not enough information to perform EDA. Check field ids.')

## 5. Visualization
Visualize numeric field distribution and relationship to a group/categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # If a group_field_id with few unique values, boxplot
    if group_field_id and df[group_field_id].nunique() <= 20:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print('Visualization skipped: missing numeric/group field.')

## 6. Conclusion

- We've loaded the FAIR⁲ dataset using the mlcroissant library directly from its Croissant metadata.
- Inspected available record sets and fields (referenced by `@id`).
- Extracted data to pandas and performed basic EDA: filtering, normalization, and grouping.
- Visualized distributions of key numeric fields.

This workflow provides a robust starting point for deeper domain analysis, modeling, or feature engineering using mass spectrometry and protein abundance data. Make sure to refer to the Croissant field and record set `@id` values in all programmatic access for full reproducibility and shareability!